# Replication Notebook — TCN-Inception (WISDM-first)


## Paper Facts and Code Availability
- Paper DOI: https://doi.org/10.1016/j.future.2024.05.023
- Official repo: none declared
- Original hyperparameters include max kernel 68 and TCN dilations (1,2,4,8).

## WISDM Adaptation
- Active config: `configs/papers/tcn_inception_wisdm.yaml`
- This notebook runs WISDM-first adaptation while preserving paper architecture intent.


## Notebook Contract
- Runtime / dependency guard
- Paper facts + code availability
- WISDM adaptation section
- FP32 training/eval on `random_stratified` + `user_holdout`
- PTQ INT8 + QAT INT8 exports
- Strict deploy-gate summary (`full_integer_io`, `tflm_compatible`, unsupported ops)
- Per-paper CSV/MD + master results append


## Compression Scope Note
- KD-related methodology from papers is documented for reproducibility context only.
- Notebook execution enforces PTQ/QAT compression flow only (`experiment.compression_focus=ptq_qat_only`).


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from src.utils.config import load_yaml
from src.run_paper_experiment import run_paper_experiment
from src.utils.device_runtime import runtime_device_report
RUN_MODE = "sanity_check"  # switch to "full_run" for final CPU-only benchmark runs
REPO_ROOT = Path.cwd().resolve()
CONFIG_PATH = REPO_ROOT / 'configs/papers/tcn_inception_wisdm.yaml'
assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"
cfg = load_yaml(CONFIG_PATH)
cfg.setdefault("runtime", {})["run_mode"] = RUN_MODE
cfg.setdefault("experiment", {})["compression_focus"] = "ptq_qat_only"
runtime = runtime_device_report(cfg)
print("paper_slug:", cfg.get("experiment", {}).get("paper_slug"))
print("model_variant:", cfg.get("experiment", {}).get("model_variant"))
print("window_size:", cfg.get("paper_protocol", {}).get("wisdm_window_override"))
print("run_mode:", runtime["run_mode"])
print("detected_gpus:", runtime["gpus"])
print("stage_device_map:", runtime["resolved_stage_devices"])


In [ ]:
# Run FP32 + PTQ + QAT + standardized reporting
out = run_paper_experiment(cfg)
out


In [ ]:
# Run-level summary table
rows_df = pd.DataFrame(out["rows"])
display(rows_df)
# Per-protocol FP32/PTQ/QAT comparison table
comp_rows = []
for r in out["rows"]:
    comp_rows.append({
        "protocol": r.get("protocol"),
        "tier": "FP32",
        "accuracy": r.get("accuracy"),
        "macro_f1": r.get("macro_f1"),
        "model_size_kb": None,
        "training_time_sec": r.get("fp32_training_time_sec"),
        "inference_latency_ms_median": None,
        "inference_latency_ms_p95": None,
    })
    comp_rows.append({
        "protocol": r.get("protocol"),
        "tier": "PTQ INT8",
        "accuracy": r.get("ptq_accuracy"),
        "macro_f1": r.get("ptq_macro_f1"),
        "model_size_kb": r.get("ptq_model_size_kb"),
        "training_time_sec": None,
        "inference_latency_ms_median": r.get("ptq_inference_latency_ms_median"),
        "inference_latency_ms_p95": r.get("ptq_inference_latency_ms_p95"),
    })
    comp_rows.append({
        "protocol": r.get("protocol"),
        "tier": "QAT INT8",
        "accuracy": r.get("qat_accuracy"),
        "macro_f1": r.get("qat_macro_f1"),
        "model_size_kb": r.get("qat_model_size_kb"),
        "training_time_sec": r.get("qat_training_time_sec"),
        "inference_latency_ms_median": r.get("qat_inference_latency_ms_median"),
        "inference_latency_ms_p95": r.get("qat_inference_latency_ms_p95"),
    })
comp_df = pd.DataFrame(comp_rows)
display(comp_df)
# Curves + confusion matrices
for r in out["rows"]:
    display(Markdown(f"### Protocol: `{r.get('protocol')}`"))
    for label, key in [
        ("FP32 training curve", "fp32_curve_png"),
        ("QAT training curve", "qat_curve_png"),
        ("FP32 confusion", "fp32_confusion_plot"),
        ("PTQ confusion", "ptq_confusion_plot"),
        ("QAT confusion", "qat_confusion_plot"),
    ]:
        p = r.get(key)
        if p and Path(p).exists():
            display(Markdown(f"**{label}**"))
            display(Image(filename=str(p), width=760))
# Condensed classification report display
for r in out["rows"]:
    display(Markdown(f"### Classification Report Snippets — `{r.get('protocol')}`"))
    for tier, key in [
        ("FP32", "fp32_metrics_json"),
        ("PTQ", "ptq_metrics_json"),
        ("QAT", "qat_metrics_json"),
    ]:
        metrics_path = r.get(key)
        if not metrics_path or not Path(metrics_path).exists():
            continue
        payload = json.loads(Path(metrics_path).read_text(encoding="utf-8"))
        cr = payload.get("classification_report", {})
        macro = cr.get("macro avg", {}) if isinstance(cr, dict) else {}
        weighted = cr.get("weighted avg", {}) if isinstance(cr, dict) else {}
        print(f"{tier}: macro_f1={macro.get('f1-score')}, weighted_f1={weighted.get('f1-score')}")
# Paper comparison exports (table + charts)
cmp = out.get("comparison_exports", {})
if cmp.get("csv") and Path(cmp["csv"]).exists():
    display(Markdown("## Paper Comparison Table"))
    display(pd.read_csv(cmp["csv"]))
for key in ["accuracy_png", "size_png", "latency_png"]:
    p = cmp.get(key)
    if p and Path(p).exists():
        display(Image(filename=str(p), width=920))
# Master aggregate snapshot
master_csv = Path(cfg["paths"]["reports_dir"]) / "results_master.csv"
if master_csv.exists():
    df_master = pd.read_csv(master_csv)
    display(Markdown("## Master Results Snapshot"))
    display(df_master[df_master["paper_slug"] == cfg["experiment"]["paper_slug"]])
print("Saved outputs root:", Path(cfg["paths"]["reports_dir"]) / cfg["experiment"]["paper_slug"])


## Notes
- If QAT conversion fails strict deploy-gate checks, results are still exported with explicit failure reason.
- Document unresolved paper ambiguities in `reports/paper_specs_<slug>.md`.
